### Phase 1: Data Collection, Conversion and Integration

In [1]:
import pandas as pd 
import numpy as np
from glob import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

Working on WGMS glaciers datasets

In [2]:
glaciers_files = glob("../final_datasets/wgms/*.xlsx")
glaciers_files

['../final_datasets/wgms\\change_band_nepal.xlsx',
 '../final_datasets/wgms\\change_nepal.xlsx',
 '../final_datasets/wgms\\event_nepal.xlsx',
 '../final_datasets/wgms\\front_variations_nepal.xlsx',
 '../final_datasets/wgms\\glaciers_nepal.xlsx',
 '../final_datasets/wgms\\mass_balance_band_nepal.xlsx',
 '../final_datasets/wgms\\mass_balance_nepal.xlsx',
 '../final_datasets/wgms\\mass_balance_point_nepal.xlsx',
 '../final_datasets/wgms\\state_nepal.xlsx']

In [3]:
for i, file_path in enumerate(glaciers_files, start=1):
    print(f"{i}: {file_path}")

    df = pd.read_excel(file_path)

    missing_percent = (
        df.isna().mean().mul(100).sort_values(ascending=False)
    )

    print("Missing percentages:\n")
    print(missing_percent)
    print(f"Shape: {df.shape}")
    print("=" * 80)


1: ../final_datasets/wgms\change_band_nepal.xlsx
Missing percentages:

remarks                 100.000000
volume_change_unc       100.000000
volume_change            99.414399
glacier_name              9.094041
glacier_id                9.094041
elevation_change_unc      1.481226
area                      1.240096
elevation_change          0.378918
change_id                 0.000000
country                   0.000000
lower_elevation           0.000000
upper_elevation           0.000000
dtype: float64
Shape: (2903, 12)
2: ../final_datasets/wgms\change_nepal.xlsx
Missing percentages:

volume_change_unc       5.38750
volume_change           5.34375
begin_outline_id        1.21250
end_outline_id          1.21250
agencies                0.73750
investigators           0.71875
elevation_change_unc    0.61875
area                    0.45625
remarks                 0.43125
glacier_name            0.22500
glacier_id              0.22500
begin_method            0.20000
begin_platform          0.

In [4]:

from pathlib import Path
import json

WGMS_DIR = Path("../final_datasets/wgms")
WGMS_OUTPUT = WGMS_DIR / "wgms_glacier_dataset.csv"

def read_wgms(name):
    path = WGMS_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing WGMS input: {path}")
    frame = pd.read_excel(path)
    frame.columns = frame.columns.str.strip().str.lower()
    return frame

with (WGMS_DIR / "datapackage.json").open(encoding="utf-8") as metadata_file:
    wgms_metadata = json.load(metadata_file)

glaciers = read_wgms("glaciers_nepal.xlsx").rename(columns={"id": "glacier_id", "short_name": "glacier_name", "gtng_region": "region"})
glaciers = glaciers[[c for c in ["glacier_id", "glacier_name", "country", "latitude", "longitude", "region", "rgi60_ids"] if c in glaciers]]
glaciers["glacier_id"] = pd.to_numeric(glaciers["glacier_id"], errors="coerce")
glaciers = glaciers.dropna(subset=["glacier_id"]).drop_duplicates("glacier_id")

def with_year(frame, date_columns):
    result = frame.copy()
    year = pd.Series(pd.NA, index=result.index, dtype="Int64")
    for column in date_columns:
        if column in result:
            year = year.fillna(pd.to_datetime(result[column], errors="coerce").dt.year.astype("Int64"))
    if "year" in result:
        year = year.fillna(pd.to_numeric(result["year"], errors="coerce").astype("Int64"))
    result["year"] = year
    return result

def numeric(frame, columns):
    for column in columns:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame

tables = {}
mb = numeric(with_year(read_wgms("mass_balance_nepal.xlsx"), ["begin_date", "end_date"]), ["annual_balance", "annual_balance_unc", "area", "ela", "ela_position", "aar"])
tables["mass_balance"] = mb.groupby(["glacier_id", "year"], as_index=False).agg(
    annual_mass_balance=("annual_balance", "mean"), annual_mass_balance_unc=("annual_balance_unc", "mean"),
    mass_balance_area=("area", "mean"), ela=("ela", "mean"), ela_position=("ela_position", "mean"), aar=("aar", "mean"))
band = numeric(read_wgms("mass_balance_band_nepal.xlsx"), ["annual_balance", "annual_balance_unc", "area"])
tables["mass_balance_band"] = band.groupby(["glacier_id", "year"], as_index=False).agg(
    band_annual_mass_balance=("annual_balance", "mean"), band_annual_mass_balance_unc=("annual_balance_unc", "mean"), band_area=("area", "mean"))
point = numeric(read_wgms("mass_balance_point_nepal.xlsx"), ["balance", "balance_unc", "elevation"])
tables["mass_balance_point"] = point.groupby(["glacier_id", "year"], as_index=False).agg(
    point_balance=("balance", "mean"), point_balance_unc=("balance_unc", "mean"), point_elevation=("elevation", "mean"))

change = numeric(with_year(read_wgms("change_nepal.xlsx"), ["end_date", "begin_date"]), ["area", "elevation_change", "elevation_change_unc", "volume_change", "volume_change_unc"])
tables["change"] = change.groupby(["glacier_id", "year"], as_index=False).agg(
    area_change=("area", "mean"), elevation_change=("elevation_change", "mean"), elevation_change_unc=("elevation_change_unc", "mean"),
    volume_change=("volume_change", "mean"), volume_change_unc=("volume_change_unc", "mean"))
front = numeric(with_year(read_wgms("front_variations_nepal.xlsx"), ["end_date", "begin_date"]), ["length_change", "length_change_unc"])
tables["front"] = front.groupby(["glacier_id", "year"], as_index=False).agg(length_change=("length_change", "mean"), length_change_unc=("length_change_unc", "mean"))
state = numeric(with_year(read_wgms("state_nepal.xlsx"), ["date"]), ["area", "length", "lowest_elevation", "mean_elevation", "highest_elevation"])
tables["state"] = state.groupby(["glacier_id", "year"], as_index=False).agg(
    state_area=("area", "mean"), glacier_length=("length", "mean"), lowest_elevation=("lowest_elevation", "mean"),
    mean_elevation=("mean_elevation", "mean"), highest_elevation=("highest_elevation", "mean"))
event = with_year(read_wgms("event_nepal.xlsx"), ["date"])
event_columns = [c for c in ["surge", "calving", "flood", "avalanche", "rockfall", "debris_flow", "earthquake", "volcanic_eruption", "other"] if c in event]
tables["event"] = event.groupby(["glacier_id", "year"], as_index=False)[event_columns].max() if event_columns else event[["glacier_id", "year"]].drop_duplicates()

index = pd.concat([t[["glacier_id", "year"]] for t in tables.values()], ignore_index=True).dropna().drop_duplicates()
index["glacier_id"] = pd.to_numeric(index["glacier_id"], errors="coerce")
index["year"] = pd.to_numeric(index["year"], errors="coerce").astype("Int64")
combined_wgms = index.merge(glaciers, on="glacier_id", how="left", validate="many_to_one")
for table in tables.values():
    combined_wgms = combined_wgms.merge(table, on=["glacier_id", "year"], how="left", validate="one_to_one")
combined_wgms["country"] = combined_wgms["country"].fillna("NP")
combined_wgms = combined_wgms.sort_values(["glacier_id", "year"], kind="stable").reset_index(drop=True)
if combined_wgms[["glacier_id", "year"]].duplicated().any():
    raise ValueError("Duplicate glacier-year rows remain.")
WGMS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
# combined_wgms.to_csv(WGMS_OUTPUT, index=False)
# print(f"WGMS {wgms_metadata['version']}: saved {len(combined_wgms):,} rows and {len(combined_wgms.columns):,} columns to {WGMS_OUTPUT}")
# combined_wgms.head()

In [5]:
df = combined_wgms.sort_values('year').reset_index(drop=True)
missing_percent = df.isna().mean().mul(100).sort_values(ascending=False)
missing_threshold = 40
removed_features = missing_percent[missing_percent > missing_threshold].index.tolist()
df = df.drop(columns=removed_features)
df.to_csv(WGMS_OUTPUT, index=False)
print(f"Removed {len(removed_features)} features with more than {missing_threshold:.0f}% missing values: {removed_features}")
print(f"WGMS {wgms_metadata['version']}: saved {len(df):,} rows and {len(df.columns):,} columns to {WGMS_OUTPUT}")
df.head()


Removed 28 features with more than 40% missing values: ['ela_position', 'band_annual_mass_balance_unc', 'point_balance_unc', 'other', 'debris_flow', 'rockfall', 'volcanic_eruption', 'earthquake', 'calving', 'avalanche', 'flood', 'surge', 'mean_elevation', 'band_area', 'state_area', 'highest_elevation', 'glacier_length', 'band_annual_mass_balance', 'length_change_unc', 'point_elevation', 'point_balance', 'lowest_elevation', 'annual_mass_balance_unc', 'length_change', 'aar', 'mass_balance_area', 'ela', 'annual_mass_balance']
WGMS 2026-02-10: saved 13,043 rows and 13 columns to ..\final_datasets\wgms\wgms_glacier_dataset.csv


,glacier_id,year,glacier_name,country,latitude,longitude,region,rgi60_ids,area_change,elevation_change,elevation_change_unc,volume_change,volume_change_unc
0,3676.0,1550,MACHHAPUCHHARE,NP,28.520300,83.991997,15_south_asia_east,NaN,NaN,NaN,NaN,NaN,NaN
1,3676.0,1560,MACHHAPUCHHARE,NP,28.520300,83.991997,15_south_asia_east,NaN,NaN,NaN,NaN,NaN,NaN
2,3461.0,1929,CHUKHUNG,NP,27.879999,86.900002,15_south_asia_east,RGI60-15.03404,NaN,NaN,NaN,NaN,NaN
3,1535.0,1952,THULAGI,NP,28.480000,84.500000,15_south_asia_east,RGI60-15.05140,NaN,NaN,NaN,NaN,NaN
4,1535.0,1958,THULAGI,NP,28.480000,84.500000,15_south_asia_east,RGI60-15.05140,NaN,NaN,NaN,NaN,NaN


In [6]:
df.isna().mean().mul(100).sort_values(ascending=False)

volume_change_unc       7.153262
volume_change           7.107261
elevation_change_unc    1.855401
area_change             1.472054
elevation_change        1.134708
rgi60_ids               0.498352
latitude                0.000000
glacier_name            0.000000
year                    0.000000
glacier_id              0.000000
country                 0.000000
longitude               0.000000
region                  0.000000
dtype: float64

Combining RGI and WGMS glaciers datasets

In [7]:
rgi_data = pd.read_csv("../final_datasets/rgi_data/raw_nepal_glaciers_data.csv")  # RGI
wgms_data = pd.read_csv("../final_datasets/wgms/wgms_glacier_dataset.csv")  # WGMS

In [8]:
rgi_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6918 entries, 0 to 6917
Data columns (total 29 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   rgi_id      6918 non-null   str    
 1   o1region    6918 non-null   int64  
 2   o2region    6918 non-null   str    
 3   glims_id    6918 non-null   str    
 4   anlys_id    6918 non-null   int64  
 5   subm_id     6918 non-null   int64  
 6   src_date    6918 non-null   str    
 7   cenlon      6918 non-null   float64
 8   cenlat      6918 non-null   float64
 9   utm_zone    6918 non-null   int64  
 10  area_km2    6918 non-null   float64
 11  primeclass  6918 non-null   int64  
 12  conn_lvl    6918 non-null   int64  
 13  surge_type  6918 non-null   int64  
 14  term_type   6918 non-null   int64  
 15  glac_name   174 non-null    str    
 16  is_rgi6     6918 non-null   int64  
 17  termlon     6918 non-null   float64
 18  termlat     6918 non-null   float64
 19  zmin_m      6918 non-null   float64
 2

In [9]:
rgi_data.head()

,rgi_id,o1region,o2region,glims_id,anlys_id,subm_id,src_date,cenlon,cenlat,utm_zone,...,zmin_m,zmax_m,zmed_m,zmean_m,slope_deg,aspect_deg,aspect_sec,dem_source,lmax_m,country
0,RGI2000-v7.0-G-15-01294,15,15-01,G080068E30442N,881651,752,2002-07-10T00:00:00,80.067137,30.441130,44,...,4111.3670,5918.4630,4652.5870,4794.8460,20.879232,7.867698,1,COPDEM30,6571,Nepal
1,RGI2000-v7.0-G-15-01295,15,15-01,G080085E30432N,881761,752,2002-07-10T00:00:00,80.084600,30.431591,44,...,5417.3223,5823.3850,5706.2080,5674.7104,34.334000,116.837890,4,COPDEM30,570,Nepal
2,RGI2000-v7.0-G-15-01296,15,15-01,G080091E30439N,881796,752,1994-09-23T00:00:00,80.092262,30.439446,44,...,4840.8440,5568.9395,5258.7470,5235.7817,40.238647,34.956500,2,COPDEM30,914,Nepal
3,RGI2000-v7.0-G-15-01297,15,15-01,G080076E30425N,881679,752,2002-07-10T00:00:00,80.071912,30.423611,44,...,5018.7104,6040.0312,5594.4960,5603.7290,35.693207,105.575230,3,COPDEM30,1654,Nepal
4,RGI2000-v7.0-G-15-01298,15,15-01,G080057E30402N,881745,752,2002-07-10T00:00:00,80.082167,30.408807,44,...,4216.8510,6034.8430,5393.9507,5219.5470,17.973806,101.787590,3,COPDEM30,8463,Nepal


In [10]:
wgms_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 13043 entries, 0 to 13042
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   glacier_id            13043 non-null  float64
 1   year                  13043 non-null  int64  
 2   glacier_name          13043 non-null  str    
 3   country               13043 non-null  str    
 4   latitude              13043 non-null  float64
 5   longitude             13043 non-null  float64
 6   region                13043 non-null  str    
 7   rgi60_ids             12978 non-null  str    
 8   area_change           12851 non-null  float64
 9   elevation_change      12895 non-null  float64
 10  elevation_change_unc  12801 non-null  float64
 11  volume_change         12116 non-null  float64
 12  volume_change_unc     12110 non-null  float64
dtypes: float64(8), int64(1), str(4)
memory usage: 1.3 MB


In [11]:
wgms_data.head()

,glacier_id,year,glacier_name,country,latitude,longitude,region,rgi60_ids,area_change,elevation_change,elevation_change_unc,volume_change,volume_change_unc
0,3676.0,1550,MACHHAPUCHHARE,NP,28.520300,83.991997,15_south_asia_east,NaN,NaN,NaN,NaN,NaN,NaN
1,3676.0,1560,MACHHAPUCHHARE,NP,28.520300,83.991997,15_south_asia_east,NaN,NaN,NaN,NaN,NaN,NaN
2,3461.0,1929,CHUKHUNG,NP,27.879999,86.900002,15_south_asia_east,RGI60-15.03404,NaN,NaN,NaN,NaN,NaN
3,1535.0,1952,THULAGI,NP,28.480000,84.500000,15_south_asia_east,RGI60-15.05140,NaN,NaN,NaN,NaN,NaN
4,1535.0,1958,THULAGI,NP,28.480000,84.500000,15_south_asia_east,RGI60-15.05140,NaN,NaN,NaN,NaN,NaN


In [12]:
import geopandas as gpd


In [13]:

# storing in standard GPS coordinates
gdf_wgms = gpd.GeoDataFrame(
    wgms_data, 
    geometry=gpd.points_from_xy(wgms_data['longitude'], wgms_data['latitude']),
    crs="EPSG:4326"
)

gdf_rgi = gpd.GeoDataFrame(
    rgi_data, 
    geometry=gpd.points_from_xy(rgi_data['cenlon'], rgi_data['cenlat']),
    crs="EPSG:4326"
)

# finding the closest RGI glacier for every WGMS record, bypassing the need for exact coordinate matches
merged_data = gpd.sjoin_nearest(
    gdf_wgms, 
    gdf_rgi, 
    how='inner', 
    distance_col='distance_degrees' # Creates a column showing how far apart the matches are
)


# keeping 0.05 degrees is roughly 5.5 kilometers, otherwise dropping them
merged_data = merged_data[merged_data['distance_degrees'] < 0.05]

# converting back to standard pandas dataframe if you don't need geometry anymore
final_df = pd.DataFrame(merged_data.drop(columns=['geometry']))

print(final_df.shape)

(13039, 44)


In [14]:
final_df.isna().mean().mul(100).sort_values(ascending=False)

glac_name               97.844927
volume_change_unc        7.155457
volume_change            7.109441
elevation_change_unc     1.855971
area_change              1.472506
elevation_change         1.135056
rgi60_ids                0.498504
glacier_name             0.000000
country_left             0.000000
region                   0.000000
year                     0.000000
glacier_id               0.000000
longitude                0.000000
latitude                 0.000000
rgi_id                   0.000000
index_right              0.000000
o2region                 0.000000
glims_id                 0.000000
anlys_id                 0.000000
o1region                 0.000000
src_date                 0.000000
cenlon                   0.000000
cenlat                   0.000000
utm_zone                 0.000000
area_km2                 0.000000
primeclass               0.000000
conn_lvl                 0.000000
subm_id                  0.000000
surge_type               0.000000
term_type     

In [15]:
# saving the combination of WGMS and RGI glacier datasets
final_df.to_csv("../final_datasets/combined_datasets/glaciers_rgi_wgms_dataset.csv")

Working on ERA5 datasets

In [16]:
import xarray as xr

In [17]:
era5_files = glob("../final_datasets/era5/*.nc")
era5_files

['../final_datasets/era5\\data_stream-moda.nc']

In [18]:
for i in range(len(era5_files)):
    print(f'{i+1}: {era5_files[i]}')
    with xr.open_dataset(era5_files[i], engine='netcdf4') as f:
        data = f.to_dataframe().reset_index()
        
        missing_percent = (
            data.isna().mean().mul(100).sort_values(ascending=False)
        )
        
        print("Missing percentages:\n")
        print(missing_percent)
        print(f"Shape: {data.shape}")
        
        print("="*80)


1: ../final_datasets/era5\data_stream-moda.nc
Missing percentages:

valid_time    0.0
latitude      0.0
longitude     0.0
d2m           0.0
t2m           0.0
skt           0.0
stl1          0.0
stl2          0.0
stl3          0.0
stl4          0.0
lblt          0.0
licd          0.0
lict          0.0
lmld          0.0
lmlt          0.0
lshf          0.0
ltlt          0.0
asn           0.0
snowc         0.0
rsn           0.0
sde           0.0
sd            0.0
sf            0.0
smlt          0.0
tsn           0.0
fal           0.0
slhf          0.0
ssr           0.0
str           0.0
sshf          0.0
ssrd          0.0
strd          0.0
ro            0.0
es            0.0
sro           0.0
e             0.0
u10           0.0
v10           0.0
sp            0.0
tp            0.0
number        0.0
expver        0.0
dtype: float64
Shape: (3168480, 42)


converting NETCDF into CSV file

In [19]:


from pathlib import Path

KEY_COLUMNS = ["time", "latitude", "longitude"]
SPATIAL_KEYS = ["latitude", "longitude"]
ERA5_OUTPUT = Path("../final_datasets/era5/era5.csv")


def _normalise_feature_names(frame, source_name, used_names):
    """Make feature names unique while leaving shared coordinate keys unchanged."""
    renamed = {}
    for column in frame.columns:
        if column in KEY_COLUMNS:
            continue
        candidate = column
        if candidate in used_names:
            candidate = f"{Path(source_name).stem}_{candidate}"
        suffix = 2
        base_candidate = candidate
        while candidate in used_names:
            candidate = f"{base_candidate}_{suffix}"
            suffix += 1
        renamed[column] = candidate
        used_names.add(candidate)
    return frame.rename(columns=renamed)


era5_paths = sorted(Path(path) for path in era5_files) if era5_files else sorted(
    Path(path) for path in glob("../final_datasets/era5/*.nc")
)
if not era5_paths:
    raise FileNotFoundError("No ERA5 .nc files were found in ../final_datasets/era5.")

frames = []
for path in era5_paths:
    with xr.open_dataset(path, engine="netcdf4") as dataset:
        has_valid_time = "valid_time" in dataset.coords
        frame = dataset[list(dataset.data_vars)].to_dataframe().reset_index()

    if "valid_time" in frame.columns:
        frame = frame.rename(columns={"valid_time": "time"})
    missing_keys = [column for column in KEY_COLUMNS if column not in frame.columns]
    if missing_keys:
        raise ValueError(f"{path.name} is missing required coordinates: {missing_keys}")
    frame["time"] = pd.to_datetime(frame["time"], errors="raise")
    frames.append((path.name, has_valid_time, frame))

dynamic_items = [item for item in frames if item[1]]
if len(dynamic_items) != 1:
    raise ValueError("Expected exactly one time-varying ERA5 file.")

dynamic_name, _, combined = dynamic_items[0]
used_features = set(combined.columns) - set(KEY_COLUMNS)

for source_name, _, spatial_frame in frames:
    if source_name == dynamic_name:
        continue
    spatial_frame = spatial_frame.drop(columns=["time"])
    spatial_frame = _normalise_feature_names(spatial_frame, source_name, used_features)
    spatial_frame = spatial_frame.drop_duplicates(subset=SPATIAL_KEYS)
    combined = combined.merge(
        spatial_frame,
        on=SPATIAL_KEYS,
        how="left",
        validate="many_to_one",
    )

combined = combined.sort_values(KEY_COLUMNS, kind="stable").reset_index(drop=True)
if combined.columns.duplicated().any():
    raise ValueError("Duplicate columns remain after merging ERA5 files.")
if combined[KEY_COLUMNS].duplicated().any():
    raise ValueError("Duplicate time/latitude/longitude rows remain after merging ERA5 files.")

ERA5_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 3168480 entries, 0 to 3168479
Data columns (total 42 columns):
 #   Column     Dtype         
---  ------     -----         
 0   time       datetime64[ns]
 1   latitude   float64       
 2   longitude  float64       
 3   d2m        float32       
 4   t2m        float32       
 5   skt        float32       
 6   stl1       float32       
 7   stl2       float32       
 8   stl3       float32       
 9   stl4       float32       
 10  lblt       float32       
 11  licd       float32       
 12  lict       float32       
 13  lmld       float32       
 14  lmlt       float32       
 15  lshf       float32       
 16  ltlt       float32       
 17  asn        float32       
 18  snowc      float32       
 19  rsn        float32       
 20  sde        float32       
 21  sd         float32       
 22  sf         float32       
 23  smlt       float32       
 24  tsn        float32       
 25  fal        float32       
 26  slhf       float32       

In [20]:
# saving ERA5 datasets
combined.to_csv(ERA5_OUTPUT, index=False)

print(f"Saved {len(combined):,} rows and {len(combined.columns):,} columns to {ERA5_OUTPUT}")
print("Columns:", list(combined.columns))
combined.head()

Saved 3,168,480 rows and 42 columns to ..\final_datasets\era5\era5.csv
Columns: ['time', 'latitude', 'longitude', 'd2m', 't2m', 'skt', 'stl1', 'stl2', 'stl3', 'stl4', 'lblt', 'licd', 'lict', 'lmld', 'lmlt', 'lshf', 'ltlt', 'asn', 'snowc', 'rsn', 'sde', 'sd', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'ro', 'es', 'sro', 'e', 'u10', 'v10', 'sp', 'tp', 'number', 'expver']


,time,latitude,longitude,d2m,t2m,skt,stl1,stl2,stl3,stl4,...,ro,es,sro,e,u10,v10,sp,tp,number,expver
0,1950-01-01,26.5,80.4,279.021973,290.124756,290.092072,291.049255,291.404358,292.600464,295.365723,...,0.000290,0.0,0.000005,-0.001282,0.151825,-0.115239,100003.0,0.000961,0,0001
1,1950-01-01,26.5,80.5,278.986816,290.099365,290.041290,290.924255,291.281311,292.475464,295.234863,...,0.000286,0.0,0.000006,-0.001222,0.181610,-0.133794,99997.0,0.001077,0,0001
2,1950-01-01,26.5,80.6,278.934082,290.066162,289.980743,290.857849,291.214905,292.403198,295.145020,...,0.000285,0.0,0.000007,-0.001227,0.217743,-0.153813,99971.0,0.001210,0,0001
3,1950-01-01,26.5,80.7,278.885254,290.048584,289.957306,290.853943,291.210999,292.391479,295.117676,...,0.000289,0.0,0.000008,-0.001256,0.251434,-0.174321,99950.0,0.001324,0,0001
4,1950-01-01,26.5,80.8,278.848145,290.066162,289.980743,290.867615,291.232483,292.440308,295.182129,...,0.000308,0.0,0.000008,-0.001221,0.279755,-0.199712,99970.0,0.001357,0,0001


Now, combining ERA5 and glaciers (RGI + WGMS) datasets

In [21]:
era5_df = pd.read_csv("../final_datasets/era5/era5.csv")
glaciers_df = pd.read_csv("../final_datasets/combined_datasets/glaciers_rgi_wgms_dataset.csv")

In [22]:
print(f"ERA5 shape: {era5_df.shape}")
print(f"WGMS Glaciers shape: {glaciers_df.shape}")

ERA5 shape: (3168480, 42)
WGMS Glaciers shape: (13039, 45)


In [23]:
era5_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3168480 entries, 0 to 3168479
Data columns (total 42 columns):
 #   Column     Dtype  
---  ------     -----  
 0   time       str    
 1   latitude   float64
 2   longitude  float64
 3   d2m        float64
 4   t2m        float64
 5   skt        float64
 6   stl1       float64
 7   stl2       float64
 8   stl3       float64
 9   stl4       float64
 10  lblt       float64
 11  licd       float64
 12  lict       float64
 13  lmld       float64
 14  lmlt       float64
 15  lshf       float64
 16  ltlt       float64
 17  asn        float64
 18  snowc      float64
 19  rsn        float64
 20  sde        float64
 21  sd         float64
 22  sf         float64
 23  smlt       float64
 24  tsn        float64
 25  fal        float64
 26  slhf       float64
 27  ssr        float64
 28  str        float64
 29  sshf       float64
 30  ssrd       float64
 31  strd       float64
 32  ro         float64
 33  es         float64
 34  sro        float64
 35  e    

In [24]:
glaciers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13039 entries, 0 to 13038
Data columns (total 45 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            13039 non-null  int64  
 1   glacier_id            13039 non-null  float64
 2   year                  13039 non-null  int64  
 3   glacier_name          13039 non-null  str    
 4   country_left          13039 non-null  str    
 5   latitude              13039 non-null  float64
 6   longitude             13039 non-null  float64
 7   region                13039 non-null  str    
 8   rgi60_ids             12974 non-null  str    
 9   area_change           12847 non-null  float64
 10  elevation_change      12891 non-null  float64
 11  elevation_change_unc  12797 non-null  float64
 12  volume_change         12112 non-null  float64
 13  volume_change_unc     12106 non-null  float64
 14  index_right           13039 non-null  int64  
 15  rgi_id                13039 no

In [25]:
# Combine annual ERA5 climate summaries with the filtered WGMS glacier-year table.
from pathlib import Path
import numpy as np

PROJECT_DIR = Path("..")
ERA5_INPUT = PROJECT_DIR / "final_datasets" / "era5" / "era5.csv"
GLACIERS_INPUT = PROJECT_DIR / "final_datasets" / "combined_datasets" / "glaciers_rgi_wgms_dataset.csv"
COMBINED_DIR = PROJECT_DIR / "final_datasets" / "combined_datasets"
COMBINED_OUTPUT = COMBINED_DIR / "era5_glaciers_datasets.csv"

era5_raw = pd.read_csv(ERA5_INPUT)
glaciers_raw = pd.read_csv(GLACIERS_INPUT)
era5_raw["time"] = pd.to_datetime(era5_raw["time"], errors="coerce")
era5_raw["year"] = era5_raw["time"].dt.year.astype("Int64")


era5_latitudes = np.sort(era5_raw["latitude"].dropna().unique())
era5_longitudes = np.sort(era5_raw["longitude"].dropna().unique())
glaciers_raw["era5_latitude"] = glaciers_raw["latitude"].map(
    lambda value: era5_latitudes[np.abs(era5_latitudes - value).argmin()]
)
glaciers_raw["era5_longitude"] = glaciers_raw["longitude"].map(
    lambda value: era5_longitudes[np.abs(era5_longitudes - value).argmin()]
)

climate_columns = [
    column for column in era5_raw.columns
    if column not in {"time", "year", "latitude", "longitude", "number", "expver"}
    and pd.api.types.is_numeric_dtype(era5_raw[column])
]
era5_annual = (
    era5_raw.groupby(["year", "latitude", "longitude"], as_index=False)[climate_columns]
    .mean()
)

combined_era5_glaciers = glaciers_raw.merge(
    era5_annual,
    left_on=["year", "era5_latitude", "era5_longitude"],
    right_on=["year", "latitude", "longitude"],
    how="inner",
    suffixes=("_wgms", "_era5"),
    validate="many_to_one",
)
combined_era5_glaciers = combined_era5_glaciers.drop(
    columns=["latitude_era5", "longitude_era5", "era5_latitude", "era5_longitude", 
             "latitude_wgms", "longitude_wgms",  "src_date", "is_rgi6", "termlon", "termlat",
             "rgi60_ids", "o1region", "o2region", "glims_id", "anlys_id", "subm_id"], errors="ignore"
)
COMBINED_DIR.mkdir(parents=True, exist_ok=True)
combined_era5_glaciers.to_csv(COMBINED_OUTPUT, index=False)

print(f"ERA5 annual grid rows: {len(era5_annual):,}")
print(f"Combined ERA5-GLACIERS rows: {len(combined_era5_glaciers):,}")
print(f"Combined columns: {len(combined_era5_glaciers.columns):,}")
print(f"Matched years: {combined_era5_glaciers['year'].min()}?{combined_era5_glaciers['year'].max()}")
print(f"Saved to: {COMBINED_OUTPUT}")
combined_era5_glaciers.head()


ERA5 annual grid rows: 265,188
Combined ERA5-GLACIERS rows: 13,036
Combined columns: 70
Matched years: 1952?2025
Saved to: ..\final_datasets\combined_datasets\era5_glaciers_datasets.csv


,Unnamed: 0,glacier_id,year,glacier_name,country_left,region,area_change,elevation_change,elevation_change_unc,volume_change,...,ssrd,strd,ro,es,sro,e,u10,v10,sp,tp
0,3,1535.0,1952,THULAGI,NP,15_south_asia_east,NaN,NaN,NaN,NaN,...,1.781730e+07,2.244371e+07,0.007598,-0.000026,0.004997,-0.000469,0.033353,-0.346741,60058.849667,0.008513
1,4,1535.0,1958,THULAGI,NP,15_south_asia_east,NaN,NaN,NaN,NaN,...,1.885359e+07,2.251194e+07,0.007271,-0.000025,0.004747,-0.000652,0.057779,-0.192186,60116.391500,0.007680
2,5,1516.0,1958,RIKHA SAMBA,NP,15_south_asia_east,NaN,NaN,NaN,NaN,...,2.076250e+07,2.047901e+07,0.003227,-0.000061,0.002977,-0.000167,0.334350,-0.168912,52253.891500,0.003743
3,6,1069.0,1970,GYAJO,NP,15_south_asia_east,NaN,NaN,NaN,NaN,...,2.058535e+07,2.040596e+07,0.002001,-0.000029,0.000715,-0.001346,0.364907,0.042529,60219.023250,0.003429
4,7,1535.0,1972,THULAGI,NP,15_south_asia_east,NaN,NaN,NaN,NaN,...,1.979244e+07,2.173098e+07,0.005823,-0.000034,0.003751,-0.000662,0.035227,-0.302494,60000.112667,0.006979


In [26]:
combined_era5_glaciers.info()

<class 'pandas.DataFrame'>
RangeIndex: 13036 entries, 0 to 13035
Data columns (total 70 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            13036 non-null  int64  
 1   glacier_id            13036 non-null  float64
 2   year                  13036 non-null  int64  
 3   glacier_name          13036 non-null  str    
 4   country_left          13036 non-null  str    
 5   region                13036 non-null  str    
 6   area_change           12847 non-null  float64
 7   elevation_change      12891 non-null  float64
 8   elevation_change_unc  12797 non-null  float64
 9   volume_change         12112 non-null  float64
 10  volume_change_unc     12106 non-null  float64
 11  index_right           13036 non-null  int64  
 12  rgi_id                13036 non-null  str    
 13  cenlon                13036 non-null  float64
 14  cenlat                13036 non-null  float64
 15  utm_zone              13036 no

In [27]:
combined_era5_glaciers.isna().mean().mul(100).sort_values(ascending=False)

glac_name               97.852102
volume_change_unc        7.134090
volume_change            7.088064
elevation_change_unc     1.833384
area_change              1.449831
                          ...    
e                        0.000000
u10                      0.000000
v10                      0.000000
sp                       0.000000
tp                       0.000000
Length: 70, dtype: float64